In [1]:
import os
import pandas as pd
import numpy as np
import nltk
import string
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from rank_bm25 import BM25Okapi

## Phase 1: Data Recovery and Text Preprocessing
In the original Data Mining pipeline, textual descriptions were dropped to focus on numerical analysis. 
Here, we recover the `Description` column from the raw dataset and merge it with our cleaned data.

We need to clean the natural language text by converting it to lowercase, 
removing punctuation, tokenizing the strings into individual words, and removing stop words.

In [2]:
stop_words = set(stopwords.words('english'))
def preprocess_text(text):
    if not isinstance(text, str):
        return []
    text = text.lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    tokens = word_tokenize(text)
    return [word for word in tokens if word not in stop_words]

In [3]:
# Download required NLTK data (run once)
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

DATA_DIR = '../data'
FEATURES_PATH = os.path.join(DATA_DIR, 'game_features.parquet')

# 1. Load the cleaned data and the raw data
if os.path.exists(FEATURES_PATH):
    print(f"Loading preprocessed dataset from {FEATURES_PATH}...")
    df = pd.read_parquet(FEATURES_PATH)
    print(f"Loaded {len(df)} games with precomputed tokens.")
else:
    print("No cache found - building dataset from raw files...")
    
    # Carica raw
    df_clean = pd.read_csv(os.path.join(DATA_DIR, 'cleaned_dataset.csv'))
    df_raw = pd.read_csv(os.path.join(DATA_DIR, 'DM1_game_dataset.csv'))
    df_text = df_raw[['BGGId', 'Description']]
    df = pd.merge(df_clean, df_text, on='BGGId', how='inner')
    df = df.dropna(subset=['Description']).reset_index(drop=True)
    df = df.rename(columns={'BGGId': 'ID'})

    #Tokenize        
    print("Tokenizing descriptions...")
    df['tokens'] = df['Description'].apply(preprocess_text)

    df.to_parquet(FEATURES_PATH, index=False)
    print(f"Saved dataset with tokens to {FEATURES_PATH}")

print(f"Dataset ready! Total board games: {len(df)}")

Loading preprocessed dataset from ../data/game_features.parquet...
Loaded 19728 games with precomputed tokens.
Dataset ready! Total board games: 19728


[nltk_data] Downloading package punkt to /home/tommaso/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /home/tommaso/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /home/tommaso/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


## BM25 Inverted Index Construction
We use the Okapi BM25 algorithm to build an inverted index. BM25 is the industry standard 
for lexical search, penalizing overly long documents and giving higher weight to rare terms (IDF).

In [4]:
# Extract the list of tokenized descriptions
corpus_tokens = df['tokens'].tolist()

# Initialize the BM25 model (this builds the index and calculates IDF)
bm25 = BM25Okapi(corpus_tokens)

print(f"BM25 Index successfully built for {len(corpus_tokens)} documents!")

BM25 Index successfully built for 19728 documents!


## Query Processing and Retrieval
We define a function to process user queries and retrieve the Top-K most relevant board games.

In [5]:
def search_games_bm25(query, top_k=5):
    """
    Searches the BM25 index for the most relevant games given a natural language query.
    """
    # 1. Preprocess the query using the exact same pipeline as the documents
    query_tokens = preprocess_text(query)
    
    # 2. Get the BM25 scores for all documents in the corpus
    doc_scores = bm25.get_scores(query_tokens)
    
    # 3. Add scores to the dataframe temporarily
    df['bm25_score'] = doc_scores
    
    # 4. Sort results descending by score and get the top K
    results = df.sort_values(by='bm25_score', ascending=False).head(top_k)
    
    # Return the Name, Score, and a snippet of the Description
    return results[['Name', 'bm25_score', 'Description']]

### Test the search engine

In [7]:
user_query = "A cooperative space game involving betrayal"
results = search_games_bm25(user_query, top_k=5)

print(f"--- Results for: '{user_query}' ---\n")
for index, row in results.iterrows():
    print(f"{row['Name']} (Score: {row['bm25_score']:.2f})")
    print(f"Snippet: {row['Description'][:150]}...\n")

--- Results for: 'A cooperative space game involving betrayal' ---

Betrayal at Baldur's Gate (Score: 15.25)
Snippet: description publisherthe shadow bhaal come baldurs gate summon monster horror darknessa build explore iconic city dark alley deadly catacomb work fell...

Cutthroat Caverns (Score: 14.38)
Snippet: quotwithout teamwork survive betrayal yoursquoll winquotcutthroat cavern play   round random encounter   essentially game kill steal round monster enc...

Puerto Diablo (Score: 13.60)
Snippet: setting caribbean   nation try establish island power money goal low level people try survive achieve personal gain possible game people need overcome...

The Binding of Isaac: Four Souls (Score: 13.58)
Snippet: official binding isaac multiplayer card game sacrifice betrayal hoarding player turn play loot card item kill monster yield item loot soul player end ...

SSO (Score: 13.56)
Snippet: semi coop scifi horror survival card counter game player attempt outlast challenge deck narrative